# Vietnamese Medical NER — VietMed-NER

Single self-contained notebook. Run **one experiment at a time** by setting `EXP_ID` (§1).
Pure-function test cells (§T) run on CPU without Colab/installs. See
`docs/superpowers/plans/2026-06-29-vietnamese-medical-ner.md` for the full plan.


## §0 Setup

In [ ]:
# §0 Setup  (skip the install/drive lines when developing locally on CPU)
!pip -q install "transformers==4.44.2" "datasets==2.21.0" seqeval pytorch-crf py_vncorenlp pandas
import os, random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from google.colab import drive          # Colab only
drive.mount('/content/drive')
DRIVE_DIR   = '/content/drive/MyDrive/vietmed_ner'
RESULTS_DIR = f'{DRIVE_DIR}/results'
CKPT_DIR    = f'{DRIVE_DIR}/checkpoints'
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## §1 Config — experiment matrix
Change `EXP_ID` to run a different row. Confirm the ViHealthBERT id loads (Task 1 note).

In [ ]:
# §1 Config
EXPERIMENTS = {
  "01_phobert_softmax": dict(encoder="vinai/phobert-base",   head="softmax", segment=True,  max_len=256, batch_size=16, grad_accum=1),
  "02_phobert_crf":     dict(encoder="vinai/phobert-base",   head="crf",     segment=True,  max_len=256, batch_size=16, grad_accum=1),
  "02b_phobert_crf128": dict(encoder="vinai/phobert-base",   head="crf",     segment=True,  max_len=128, batch_size=16, grad_accum=1),
  "03_xlmr_crf":        dict(encoder="xlm-roberta-base",     head="crf",     segment=False, max_len=256, batch_size=16, grad_accum=1),
  "04_vihealthbert_crf":dict(encoder="demdecuong/vihealthbert-base-word", head="crf", segment=True, max_len=256, batch_size=16, grad_accum=1),
  "05_phobert_gat_crf": dict(encoder="vinai/phobert-base",   head="gat_crf", segment=True,  max_len=128, batch_size=8,  grad_accum=2),
}
EXP_ID = "01_phobert_softmax"   # <-- change this to run a different row
cfg = EXPERIMENTS[EXP_ID]
print(EXP_ID, cfg)

## §2 Data — load VietMed-NER, derive labels, shared cross-domain schema

In [ ]:
# §2 Data
from datasets import load_dataset
raw = load_dataset("leduckhai/VietMed-NER")
for c in ("audio", "duration"):
    if c in raw["train"].column_names:
        raw = raw.remove_columns(c)

TAG_COL  = "labels" if "labels" in raw["train"].column_names else "tags"
WORD_COL = "words"

def get_words_labels(split):
    ds = raw[split]
    return list(ds[WORD_COL]), list(ds[TAG_COL])

_all_tags = set(t for split in raw for row in raw[split][TAG_COL] for t in row)
LABEL_LIST = ["O"] + sorted(t for t in _all_tags if t != "O")
label2id = {t:i for i,t in enumerate(LABEL_LIST)}
id2label = {i:t for t,i in label2id.items()}
print(len(LABEL_LIST), "labels; splits:", {k: len(raw[k]) for k in raw})

In [ ]:
# §2 (cont): shared cross-domain schema
SHARED_MAP_VIETMED = {
  "DISEASESYMPTOM":"SYMPTOM_DISEASE", "AGE":"AGE", "GENDER":"GENDER",
  "OCCUPATION":"OCCUPATION", "LOCATION":"LOCATION", "ORGANIZATION":"ORGANIZATION",
  "DATETIME":"DATE",
}
SHARED_MAP_PHONER = {
  "SYMPTOM&DISEASE":"SYMPTOM_DISEASE", "AGE":"AGE", "GENDER":"GENDER",
  "OCCUPATION":"OCCUPATION", "LOCATION":"LOCATION", "ORGANIZATION":"ORGANIZATION",
  "DATE":"DATE",
}
SHARED_LABELS = ["O"] + sorted(set(SHARED_MAP_VIETMED.values()))

def project_tags(tags, mapping):
    out = []
    for t in tags:
        if t == "O":
            out.append("O"); continue
        pref, _, typ = t.partition("-")
        out.append(f"{pref}-{mapping[typ]}" if typ in mapping else "O")
    return out

## §3 Preprocess — per-encoder segmentation, tag re-align, subword -100

In [ ]:
# §3 Preprocess: syllable->word tag re-alignment  (verified on CPU)
def realign_tags(syllables, tags, word_groups):
    words, new_tags = [], []
    for grp in word_groups:
        words.append("_".join(syllables[i] for i in grp))
        head = tags[grp[0]]
        if head.startswith("I-"):                 # word boundary fell inside an entity
            head = "B-" + head[2:]
        new_tags.append(head)
    return words, new_tags

In [ ]:
# §3 (cont): py_vncorenlp word segmentation -> word_groups
import py_vncorenlp
_SEG = None
def get_segmenter():
    global _SEG
    if _SEG is None:
        py_vncorenlp.download_model(save_dir=f'{DRIVE_DIR}/vncorenlp')
        _SEG = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=f'{DRIVE_DIR}/vncorenlp')
    return _SEG

def segment_to_groups(syllables):
    """Return word_groups: list of syllable-index lists. Fallback: 1 syllable/word."""
    seg = get_segmenter()
    segged = seg.word_segment(" ".join(syllables))
    word_units = " ".join(segged).split()
    groups, cur = [], 0
    for wu in word_units:
        n = wu.count("_") + 1
        groups.append(list(range(cur, cur + n)))
        cur += n
    if cur != len(syllables):
        return [[i] for i in range(len(syllables))]
    return groups

In [ ]:
# §3 (cont): subword -100 alignment  (verified on CPU)
def align_labels(tokenized_inputs, word_label_ids):
    aligned = []
    for i, labels in enumerate(word_label_ids):
        prev, ids = None, []
        for wid in tokenized_inputs.word_ids(batch_index=i):
            if wid is None:   ids.append(-100)
            elif wid != prev: ids.append(labels[wid])
            else:             ids.append(-100)
            prev = wid
        aligned.append(ids)
    tokenized_inputs["labels"] = aligned
    return tokenized_inputs

In [ ]:
# §3 (cont): build encoded dataset for a given cfg
from transformers import AutoTokenizer
def build_encoded(split, cfg, tokenizer):
    words_col, tags_col = get_words_labels(split)
    enc_words, enc_label_ids = [], []
    for syl, tags in zip(words_col, tags_col):
        if cfg["segment"]:
            groups = segment_to_groups(syl)
            w, t = realign_tags(syl, tags, groups)
        else:
            w, t = syl, tags
        enc_words.append(w); enc_label_ids.append([label2id[x] for x in t])
    tok = tokenizer(enc_words, is_split_into_words=True, truncation=True,
                    max_length=cfg["max_len"], padding="max_length")
    tok = align_labels(tok, enc_label_ids)
    return tok

## §4 Verify — decode-back gate (MUST pass before training)

In [ ]:
# §4 Verify (GATE)
def verify_roundtrip(cfg, tokenizer, n=5):
    tok = build_encoded("train", cfg, tokenizer)
    words_col, tags_col = get_words_labels("train")
    ok = 0
    for i in range(n):
        wids = tok.word_ids(batch_index=i); labs = tok["labels"][i]
        rec, seen = [], set()
        for wid, lab in zip(wids, labs):
            if wid is None or wid in seen: continue
            seen.add(wid); rec.append(id2label[lab])
        syl, tags = words_col[i], tags_col[i]
        exp = realign_tags(syl, tags, segment_to_groups(syl))[1] if cfg["segment"] else tags
        exp = exp[:len(rec)]
        assert rec == exp, f"sample {i} mismatch:\n  rec={rec}\n  exp={exp}"
        ok += 1
    print(f"verify_roundtrip PASSED for {ok}/{n} samples")

# tokenizer = AutoTokenizer.from_pretrained(cfg["encoder"]); verify_roundtrip(cfg, tokenizer)

## §5 Model — softmax / CRF factory

In [ ]:
# §5 Model
import torch.nn as nn
from torchcrf import CRF
from transformers import AutoModel, AutoModelForTokenClassification

class BertCrfForNer(nn.Module):
    def __init__(self, encoder_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(encoder_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
    def forward(self, input_ids, attention_mask, labels=None):
        h = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        emissions = self.classifier(self.dropout(h))
        mask = attention_mask.bool()
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            return -self.crf(emissions, safe, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)

def build_model(cfg):
    n = len(LABEL_LIST)
    if cfg["head"] == "softmax":
        return AutoModelForTokenClassification.from_pretrained(
            cfg["encoder"], num_labels=n, id2label=id2label, label2id=label2id)
    if cfg["head"] == "crf":
        return BertCrfForNer(cfg["encoder"], n)
    if cfg["head"] == "gat_crf":
        return build_gat_crf(cfg["encoder"], n)   # defined in §10
    raise ValueError(cfg["head"])

## §7 Eval — seqeval entity-level (defined before §6 so training can call it)

In [ ]:
# §7 Eval
from seqeval.metrics import f1_score, classification_report
def evaluate_tags(true_tags, pred_tags):
    return {
       "micro_f1": f1_score(true_tags, pred_tags, average="micro"),
       "macro_f1": f1_score(true_tags, pred_tags, average="macro"),
       "report":   classification_report(true_tags, pred_tags, digits=4),
    }

def decode_predictions(pred, label_ids, head):
    true_tags, pred_tags = [], []
    for i, gold in enumerate(label_ids):
        p_seq = pred[i]
        t_row, p_row, j = [], [], 0
        for k, g in enumerate(gold):
            if g == -100:
                continue
            t_row.append(id2label[g])
            p_row.append(id2label[p_seq[k]] if head == "softmax" else id2label[p_seq[j]])
            j += 1
        true_tags.append(t_row); pred_tags.append(p_row)
    return true_tags, pred_tags

## §6 Train — loop + Drive checkpoint + CSV log

In [ ]:
# §6 Train
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup

def _to_dataset(tok):
    ids  = torch.tensor(tok["input_ids"]); mask = torch.tensor(tok["attention_mask"])
    labs = torch.tensor(tok["labels"]);    return TensorDataset(ids, mask, labs)

def train_and_eval(cfg, exp_id, epochs=30, lr=2e-5, patience=3):
    tokenizer = AutoTokenizer.from_pretrained(cfg["encoder"])
    dl_tr = DataLoader(_to_dataset(build_encoded("train", cfg, tokenizer)), batch_size=cfg["batch_size"], shuffle=True)
    dl_va = DataLoader(_to_dataset(build_encoded("validation", cfg, tokenizer)), batch_size=cfg["batch_size"])
    dl_te = DataLoader(_to_dataset(build_encoded("test", cfg, tokenizer)), batch_size=cfg["batch_size"])
    model = build_model(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(dl_tr)*epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1*total), total)
    scaler = torch.cuda.amp.GradScaler()
    best_f1, bad, ckpt = -1, 0, f'{CKPT_DIR}/{exp_id}.pt'

    def run_eval(dl):
        model.eval(); T, P = [], []
        with torch.no_grad():
            for ids, mask, labs in dl:
                ids, mask = ids.to(DEVICE), mask.to(DEVICE)
                if cfg["head"] == "softmax":
                    pred = model(ids, attention_mask=mask).logits.argmax(-1).cpu().tolist()
                else:
                    pred = model(ids, mask)
                t, p = decode_predictions(pred, labs.tolist(), cfg["head"]); T += t; P += p
        return evaluate_tags(T, P)

    for ep in range(epochs):
        model.train()
        for step, (ids, mask, labs) in enumerate(dl_tr):
            ids, mask, labs = ids.to(DEVICE), mask.to(DEVICE), labs.to(DEVICE)
            with torch.cuda.amp.autocast():
                loss = model(ids, attention_mask=mask, labels=labs).loss if cfg["head"] == "softmax" else model(ids, mask, labs)
                loss = loss/cfg["grad_accum"]
            scaler.scale(loss).backward()
            if (step+1) % cfg["grad_accum"] == 0:
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        m = run_eval(dl_va); print(f"ep{ep} val micro_f1={m['micro_f1']:.4f}")
        torch.save(model.state_dict(), ckpt)
        if m["micro_f1"] > best_f1:
            best_f1 = m["micro_f1"]; bad = 0; torch.save(model.state_dict(), ckpt+".best")
        else:
            bad += 1
        if bad >= patience: print("early stop"); break

    model.load_state_dict(torch.load(ckpt+".best"))
    test_m = run_eval(dl_te)
    pd.DataFrame([{ "exp_id":exp_id, "encoder":cfg["encoder"], "head":cfg["head"],
                    "max_len":cfg["max_len"], "micro_f1":test_m["micro_f1"],
                    "macro_f1":test_m["macro_f1"]}]).to_csv(f'{RESULTS_DIR}/{exp_id}.csv', index=False)
    print(test_m["report"]); return test_m

## §8 Error analysis — boundary / type / missing / spurious

In [ ]:
# §8 Error analysis  (spans + classify_errors verified on CPU)
import pandas as pd
def spans(tags):
    out, start, typ = set(), None, None
    for i, t in enumerate(tags + ["O"]):
        if t == "O" or t.startswith("B-"):
            if start is not None: out.add((start, i, typ))
            start, typ = (None, None)
        if t.startswith("B-"): start, typ = i, t[2:]
        elif t.startswith("I-") and start is None: start, typ = i, t[2:]
    return out

def classify_errors(true_tags, pred_tags, words):
    rows = []
    for gold, pred, w in zip(true_tags, pred_tags, words):
        G, P = spans(gold), spans(pred)
        gold_by_type = {(s,e):ty for (s,e,ty) in G}; pred_by_type = {(s,e):ty for (s,e,ty) in P}
        sent = " ".join(w)
        for (s,e,ty) in G:
            if (s,e,ty) in P: continue
            if (s,e) in pred_by_type:
                rows.append(dict(category="type", sentence=sent, gold_span=f"{ty}:{w[s:e]}", pred_span=f"{pred_by_type[(s,e)]}:{w[s:e]}"))
            elif any(ps<e and s<pe and pty==ty for (ps,pe,pty) in P):
                rows.append(dict(category="boundary", sentence=sent, gold_span=f"{ty}:{w[s:e]}", pred_span="(overlap)"))
            else:
                rows.append(dict(category="missing", sentence=sent, gold_span=f"{ty}:{w[s:e]}", pred_span="-"))
        for (s,e,ty) in P:
            if (s,e,ty) in G: continue
            if (s,e) in gold_by_type: continue
            if any(gs<e and s<ge and gty==ty for (gs,ge,gty) in G): continue
            rows.append(dict(category="spurious", sentence=sent, gold_span="-", pred_span=f"{ty}:{w[s:e]}"))
    return pd.DataFrame(rows)

def error_counts(df):
    return df["category"].value_counts().to_dict()

## §9 Cross-domain — zero-shot on PhoNER-COVID19 via shared schema

In [ ]:
# §9 Cross-domain
!git clone -q https://github.com/VinAIResearch/PhoNER_COVID19.git
def read_conll(path):
    words, tags, w, t = [], [], [], []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line:
            if w: words.append(w); tags.append(t); w, t = [], []
        else:
            parts = line.split(); w.append(parts[0]); t.append(parts[-1])
    if w: words.append(w); tags.append(t)
    return words, tags
# Confirm exact path after clone:  !ls PhoNER_COVID19/data/word
ph_words, ph_tags = read_conll("PhoNER_COVID19/data/word/test_word.conll")
print(len(ph_words), "PhoNER test sentences")

In [ ]:
def crossdomain_eval(best_exp_id):
    cfg = EXPERIMENTS[best_exp_id]; tokenizer = AutoTokenizer.from_pretrained(cfg["encoder"])
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(f'{CKPT_DIR}/{best_exp_id}.pt.best')); model.eval()
    tok = tokenizer(ph_words, is_split_into_words=True, truncation=True,
                    max_length=cfg["max_len"], padding="max_length")
    ds = TensorDataset(torch.tensor(tok["input_ids"]), torch.tensor(tok["attention_mask"]))
    dl = DataLoader(ds, batch_size=cfg["batch_size"])
    preds = []
    with torch.no_grad():
        for ids, mask in dl:
            ids, mask = ids.to(DEVICE), mask.to(DEVICE)
            out = (model(ids, attention_mask=mask).logits.argmax(-1).cpu().tolist()
                   if cfg["head"] == "softmax" else model(ids, mask))
            preds.extend(out)
    T_tags, P_tags = [], []
    for i, gold in enumerate(ph_tags):
        wids = tok.word_ids(batch_index=i); seen = set(); rec = []; j = 0
        for k, wid in enumerate(wids):
            if wid is None or wid in seen:
                continue
            seen.add(wid)
            pid = preds[i][k] if cfg["head"] == "softmax" else preds[i][j]; j += 1
            rec.append(id2label[pid])
        rec = rec[:len(gold)]
        P_tags.append(project_tags(rec, SHARED_MAP_VIETMED))
        T_tags.append(project_tags(gold, SHARED_MAP_PHONER))
    m = evaluate_tags(T_tags, P_tags)
    pd.DataFrame([{ "best_exp":best_exp_id, "cross_micro_f1":m["micro_f1"], "cross_macro_f1":m["macro_f1"]}]).to_csv(f'{RESULTS_DIR}/crossdomain.csv', index=False)
    print(m["report"]); return m

## §10 GAT model — PhoBERT + GAT + CRF (improved)

In [ ]:
# §10 GAT model: install pinned to Colab CUDA, then define build_gat_crf
print(torch.__version__, torch.version.cuda)
!pip -q install torch-geometric

In [ ]:
from torch_geometric.nn import GATConv
class GatCrfForNer(nn.Module):
    def __init__(self, encoder_name, num_labels, heads=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(encoder_name)
        H = self.bert.config.hidden_size
        self.gat = GATConv(H, H//heads, heads=heads, dropout=0.1)
        self.attn = nn.MultiheadAttention(H, heads, batch_first=True)
        self.classifier = nn.Linear(H, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
    def _full_edges(self, L, device):
        idx = torch.arange(L, device=device)
        return torch.stack([idx.repeat_interleave(L), idx.repeat(L)], 0)
    def forward(self, input_ids, attention_mask, labels=None):
        h = self.bert(input_ids, attention_mask=attention_mask).last_hidden_state
        B, L, H = h.shape; ei = self._full_edges(L, h.device)
        g = torch.stack([self.gat(h[b], ei) for b in range(B)], 0)
        a, _ = self.attn(g, g, g, key_padding_mask=~attention_mask.bool())
        emissions = self.classifier(a); mask = attention_mask.bool()
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            return -self.crf(emissions, safe, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)

def build_gat_crf(encoder_name, num_labels): return GatCrfForNer(encoder_name, num_labels)

## §T Tests — CPU-only, no Colab/installs needed
These verify the bug-prone pure functions. Run after defining §2–§3, §8.

In [ ]:
# §T tests (pure functions — verified on CPU)
def _test_project():
    m = {"DISEASESYMPTOM":"SYMPTOM_DISEASE", "DATETIME":"DATE"}
    assert project_tags(["B-DISEASESYMPTOM","I-DISEASESYMPTOM","O"], m) == ["B-SYMPTOM_DISEASE","I-SYMPTOM_DISEASE","O"]
    assert project_tags(["B-DRUGCHEMICAL","I-DRUGCHEMICAL"], m) == ["O","O"]
    assert project_tags(["B-DATETIME"], m) == ["B-DATE"]
    print("project_tags OK")

def _test_realign():
    syl=["bệnh","nhân","bị","viêm","phổi"]; tags=["B-DISEASESYMPTOM","I-DISEASESYMPTOM","O","B-DISEASESYMPTOM","I-DISEASESYMPTOM"]
    w,t = realign_tags(syl, tags, [[0,1],[2],[3,4]])
    assert w == ["bệnh_nhân","bị","viêm_phổi"] and t == ["B-DISEASESYMPTOM","O","B-DISEASESYMPTOM"]
    assert realign_tags(["a","b"], ["O","I-AGE"], [[0],[1]])[1] == ["O","B-AGE"]
    print("realign_tags OK")

class _FakeTok(dict):
    def __init__(self, wid): super().__init__(); self._wid=wid
    def word_ids(self, batch_index=0): return self._wid[batch_index]
def _test_align():
    out = align_labels(_FakeTok([[None,0,0,1,None]]), [[5,7]])
    assert out["labels"][0] == [-100,5,-100,7,-100]; print("align_labels OK")

def _test_errors():
    assert spans(["B-AGE","I-AGE","O","B-LOCATION"]) == {(0,2,"AGE"),(3,4,"LOCATION")}
    assert set(classify_errors([["B-AGE","I-AGE","O"]], [["B-AGE","O","O"]], [["a","b","c"]])["category"]) == {"boundary"}
    assert set(classify_errors([["B-AGE","O"]], [["B-GENDER","O"]], [["a","b"]])["category"]) == {"type"}
    assert set(classify_errors([["B-AGE","O"]], [["O","B-AGE"]], [["a","b"]])["category"]) == {"missing","spurious"}
    print("error classify OK")

_test_project(); _test_realign(); _test_align(); _test_errors()
print("ALL §T TESTS PASSED")